In [1]:
%%markdown

# Линейные методы

## Конфигурация окружения.



# Линейные методы

## Конфигурация окружения.


In [2]:
%cd %pwd/../../

Z:\.ITMO\sem_05\andan\labs\001\src


In [3]:
%%markdown

### Библиотеки

`sklearn` - только для загрузки и разбиения датасета, ну и банальный `matplotlib` для вывода.



### Библиотеки

`sklearn` - только для загрузки и разбиения датасета, ну и банальный `matplotlib` для вывода.


In [4]:
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_svmlight_file
from sklearn.model_selection import train_test_split

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

In [5]:
from IPython.display import display
_ = get_ipython().display_formatter.formatters['text/plain'].for_type(str, lambda string, pp, _: pp.text(string))

In [6]:
%%markdown

### Загрузка датасета

Я взял датасет с курса методов оптимизации, в нём немного признаков и он разумного размера.



### Загрузка датасета

Я взял датасет с курса методов оптимизации, в нём немного признаков и он разумного размера.


In [7]:
DATA_PATH = Path("./datasets/ijcnn1.txt.bz2")

@dataclass
class Dataset:
    X: np.ndarray
    y: np.ndarray

def load_dataset(path: Path) -> Dataset:
    X_sparse, y = load_svmlight_file(path)
    return Dataset(X_sparse.toarray().astype(np.float64), y.astype(np.float64))

DATA = load_dataset(DATA_PATH)

f"""Shape {DATA.X.shape}
classes {np.unique(DATA.y)}, class balance {np.bincount((DATA.y > 0).astype(int))}"""

Shape (49990, 22)
classes [-1.  1.], class balance [45137  4853]

In [8]:
%%markdown

### Разбиение датасета и проверка



### Разбиение датасета и проверка


In [9]:
RANDOM_STATE: int = 30

def split(test_size: float, *, data: Dataset = DATA, seed: int = RANDOM_STATE) -> tuple[Dataset, Dataset]:
    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = train_test_split(
        data.X, data.y, test_size=test_size, stratify=data.y, random_state=seed,
    )

    mu = X_train.mean(axis=0)
    sigma = X_train.std(axis=0)
    sigma[sigma == 0] = 1.0

    X_train = (X_train - mu) / sigma
    X_test = (X_test - mu) / sigma

    return Dataset(X_train, y_train), Dataset(X_test, y_test)

train, test = split(0.2)

f"""Train:
  Shape {train.X.shape}, class balance {np.bincount((train.y > 0).astype(int))}
Test:
  Shape {test.X.shape}, class balance {np.bincount((test.y > 0).astype(int))}
"""

Train:
  Shape (39992, 22), class balance [36110  3882]
Test:
  Shape (9998, 22), class balance [9027  971]


In [10]:
%%markdown

### Функции качества/ошибки

Банальное количество совпадений как точность.



### Функции качества/ошибки

Банальное количество совпадений как точность.


In [11]:
def accuracy(y_true: np.ndarray, y_score: np.ndarray) -> float:
    return float(np.mean(np.sign(y_score) == y_true))

def error(y_true: np.ndarray, y_score: np.ndarray) -> float:
    return 1.0 - accuracy(y_true, y_score)